# Calibracion dinamica en GPU

Ensayo interactivo de grabacion y directo simulado. No conecta la GPU a la API local. Selecciona un entorno GPU y empieza con una sesion nueva. Usa tu video y el ZIP de codigo actualizado; no subas archivos .env ni credenciales. Las medidas 105 x 68 m deben verificarse para otro campo.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, json, hashlib, sys
from uuid import uuid4
subidos = files.upload()  # Seleccionar worker-calibracion-dinamica-2026-11-17.zip
if len(subidos) != 1:
    raise ValueError("Selecciona solo el paquete de codigo")
paquete = Path(next(iter(subidos)))
subidos.clear()
raiz = Path("/content") / ("futbol-" + uuid4().hex[:8])
raiz.mkdir()
with zipfile.ZipFile(paquete) as z:
    for nombre in z.namelist():
        if not (raiz / nombre).resolve().is_relative_to(raiz.resolve()):
            raise ValueError("Ruta de ZIP no permitida")
    z.extractall(raiz)
manifest = json.loads((raiz / "manifiesto.json").read_text())
for nombre, esperado in manifest.items():
    if hashlib.sha256((raiz / "vision" / nombre).read_bytes()).hexdigest() != esperado:
        raise ValueError("El codigo no coincide con su manifiesto")
sys.path.insert(0, str(raiz / "vision"))
print("Codigo verificado:", len(manifest), "archivos")


In [ ]:
%pip -q install -r {raiz / "vision" / "requirements.txt"}


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Selecciona GPU en la configuracion del entorno")
print("GPU:", torch.cuda.get_device_name(0))
from calibracion.instalar_modelo import instalar
print("Checkpoint de cancha verificado:", instalar())


In [ ]:
subidos_video = files.upload()  # Seleccionar el video del ensayo
if len(subidos_video) != 1:
    raise ValueError("Selecciona solo un video")
video = Path(next(iter(subidos_video)))
subidos_video.clear()
print("Video:", video.name, "bytes:", video.stat().st_size)


## Dos modalidades, mismo motor

Se ensayan 30 segundos por modalidad. En directo se descartan cuadros atrasados. Los cuadros/s de directo incluyen la espera del reloj y no equivalen a rendimiento maximo. Revisa las capturas y referencias: una ejecucion exitosa no certifica precision del balon o de la geometria.

In [ ]:
from deteccion.modelo import ConfigModelo, ModeloDeteccion, CLASES_COCO
from evaluacion.ensayo_video import ensayar
import gc
salida = Path("/content") / ("ensayo-dinamico-" + uuid4().hex[:8])
configuracion = {"modo":"AUTOMATICA", "partido_id":1, "largo_m":105, "ancho_m":68, "valida":False, "matriz_h":None}
for nombre, directo in (("grabacion", False), ("directo-simulado", True)):
    modelo = ModeloDeteccion(ConfigModelo(ruta_pesos="yolo11n.pt", clases=dict(CLASES_COCO), dispositivo="cuda:0", tamano_img=1280))
    modelo.cargar()
    resumen = ensayar(modelo, video, salida / nombre, segundos=30, stride=1, simular_directo=directo, calibracion=configuracion)
    print(nombre, resumen)
    if not str(resumen["dispositivo"]).startswith("cuda"):
        raise RuntimeError("La inferencia de deteccion no uso GPU")
    del modelo
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
import shutil
archivo = shutil.make_archive(str(salida), "zip", salida)
files.download(archivo)


## Evidencia que debes revisar

Comprueba resumen.json, muestras.json y las capturas de ambas carpetas. Conserva el ZIP para comparar tiempos, cuadros descartados, referencia disponible y observaciones de balon. Faltan la prueba de camara fisica, la conexion remota con la aplicacion y dos horas continuas en el entorno de presentacion.